# LLM Comparison: qwen2.5:7b-instruct-q4_K_M vs qwen2.5:14b-instruct 

Side-by-side extraction quality and speed comparison using the 4-agent pipeline on 5 receipts.

## Setup

In [1]:
import sys
from pathlib import Path

# Make sure the local src is on the path when running from the notebooks/ directory
repo_root = Path().resolve().parent  # finamt/notebooks -> finamt/
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import time
import warnings
import pandas as pd

warnings.filterwarnings("ignore")

from finamt import FinanceAgent
from finamt.agents.config import AgentsConfig

print("finamt imported successfully")

finamt imported successfully


In [3]:
# ── Receipt files ────────────────────────────────────────────────────────────
DATA_DIR = repo_root / "data"
receipts = sorted(DATA_DIR.glob("*.pdf"))
assert len(receipts) == 5, f"Expected 5 PDFs, found {len(receipts)}"
print("Receipts to process:")
for r in receipts:
    print(f"  {r.name}")

# ── Models to compare ────────────────────────────────────────────────────────
MODELS = {
    "qwen2.5:7b-instruct-q4_K_M": AgentsConfig(agent_model="qwen2.5:7b-instruct-q4_K_M"),
    "qwen2.5:14b-instruct":       AgentsConfig(agent_model="qwen2.5:14b-instruct"),
}

Receipts to process:
  adobe-2601.pdf
  anthropic-2601.pdf
  github-2601.pdf
  microsoft-2601.pdf
  openai-2601.pdf


## Run Extraction

Each model processes all 5 receipts sequentially using the 4-agent pipeline.  
Persistence is disabled (`db_path=None`) so no data is written to disk.

In [6]:
def _fmt_result(result) -> str:
    """Compact single-line summary of extraction result for table display."""
    if not result.success:
        return f"ERROR: {result.error_message}"
    d = result.data
    cp = d.counterparty.name if d.counterparty else "—"
    date = d.receipt_date.date().isoformat() if d.receipt_date else "—"
    total = f"{d.total_amount}" if d.total_amount is not None else "—"
    items = len(d.items) if d.items else 0
    cat = str(d.category) if d.category else "—"
    return f"{cp} | {date} | {total} EUR | {items} items | {cat}"


# ── Main extraction loop ──────────────────────────────────────────────────────
raw_results: dict[str, list[dict]] = {}   # model_name -> list of row dicts

for model_name, agents_cfg in MODELS.items():
    print(f"\n{'─'*60}")
    print(f"Model: {model_name}")
    print(f"{'─'*60}")
    agent = FinanceAgent(agents_cfg=agents_cfg, db_path=None)
    rows = []
    for pdf in receipts:
        t0 = time.perf_counter()
        result = agent.process_receipt(pdf)
        elapsed = round(time.perf_counter() - t0, 2)
        rows.append({
            "receipt":  pdf.name,
            "success":  result.success,
            "summary":  _fmt_result(result),
            "time_s":   elapsed,
        })
        status = "✓" if result.success else "✗"
        print(f"  {status}  {pdf.name:<35}  {elapsed:>6.2f}s")
    raw_results[model_name] = rows

print("\nDone.")


────────────────────────────────────────────────────────────
Model: qwen2.5:7b-instruct-q4_K_M
────────────────────────────────────────────────────────────
[finamt] adobe-2601.pdf
  [20:56:39] → PyMuPDF ...
  [20:56:39] → PDF text layer (page 1)
  [20:56:39] → Extraction pipeline ...
  [20:56:39] → Agent 1: metadata
  [20:56:43] → Agent 2: counterparty
  [20:56:49] → Agent 3: amounts
  [20:56:53] → Agent 4: line items
  ✓  adobe-2601.pdf                        17.53s
[finamt] anthropic-2601.pdf
  [20:56:57] → PyMuPDF ...
  [20:56:57] → PDF text layer (page 1)
  [20:56:57] → Extraction pipeline ...
  [20:56:57] → Agent 1: metadata
  [20:57:00] → Agent 2: counterparty
  [20:57:04] → Agent 3: amounts
  [20:57:07] → Agent 4: line items
  ✓  anthropic-2601.pdf                    13.79s
[finamt] github-2601.pdf
  [20:57:10] → PyMuPDF ...
  [20:57:11] → PDF text layer (page 1)
  [20:57:11] → Extraction pipeline ...
  [20:57:11] → Agent 1: metadata
  [20:57:14] → Agent 2: counterparty
  [20:5

## Results — Side-by-Side Comparison

In [7]:
model_names = list(MODELS.keys())
m0, m1 = model_names[0], model_names[1]

df0 = pd.DataFrame(raw_results[m0]).set_index("receipt")
df1 = pd.DataFrame(raw_results[m1]).set_index("receipt")

comparison = pd.DataFrame({
    f"{m0} — extraction":   df0["summary"],
    f"{m0} — time (s)":     df0["time_s"],
    f"{m1} — extraction":   df1["summary"],
    f"{m1} — time (s)":     df1["time_s"],
})

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 300)
comparison

,qwen2.5:7b-instruct-q4_K_M — extraction,qwen2.5:7b-instruct-q4_K_M — time (s),qwen2.5:14b-instruct — extraction,qwen2.5:14b-instruct — time (s)
receipt,,,,
adobe-2601.pdf,Adobe Systems Software Ireland Ltd | 2026-01-30 | 55.84 EUR | 1 items | software,17.53,Adobe Systems Software Ireland Ltd | 2026-01-30 | 55.84 EUR | 1 items | software,45.95
anthropic-2601.pdf,"Anthropic, PBC | 2026-01-21 | 6.0 EUR | 1 items | other",13.79,"Anthropic, PBC | 2026-01-21 | 6.0 EUR | 1 items | financial",28.07
github-2601.pdf,"GitHub, Inc. | 2026-01-11 | 10.0 EUR | 1 items | services",14.40,"GitHub, Inc. | 2026-01-11 | 10.0 EUR | 1 items | software",27.99
microsoft-2601.pdf,Microsoft Ireland Operations Ltd | 2026-01-02 | 13.92 EUR | 1 items | services,24.00,Microsoft Ireland Operations Ltd | 2026-01-02 | 13.92 EUR | 1 items | services,48.92
openai-2601.pdf,"OpenAI, LLC | 2026-01-27 | 10.0 EUR | 1 items | services",13.64,"OpenAI, LLC | 2026-01-27 | 10.0 EUR | 1 items | software",27.87


## Timing Summary

In [8]:
timing = pd.DataFrame({
    "model":       model_names,
    "total_s":     [sum(r["time_s"] for r in raw_results[m]) for m in model_names],
    "avg_s":       [sum(r["time_s"] for r in raw_results[m]) / len(receipts) for m in model_names],
    "successes":   [sum(r["success"] for r in raw_results[m]) for m in model_names],
}).set_index("model")

timing["total_s"] = timing["total_s"].round(2)
timing["avg_s"]   = timing["avg_s"].round(2)

timing

,total_s,avg_s,successes
model,,,
qwen2.5:7b-instruct-q4_K_M,83.36,16.67,5
qwen2.5:14b-instruct,178.80,35.76,5
